In [3]:
from catboost import CatBoostRegressor
from ngboost.distns import T
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from rich import print

pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv("data/model.csv")
df.head()

,set,t,rarity,pull_rate,card,number,is_pokemon,is_chase,price_on_release,current_price,popularity_index,ebay_index,set_popularity_index,next_price,next_log_return,log_return_1m,log_return_2m,log_return_3m,momentum_3m,volatility_3m,card_age,log_current_price,return_since_release,log_return_6m,momentum_6m,price_mean_3m,price_mean_6m,price_vs_mean_3m,price_vs_mean_6m,volatility_6m,history_length,popularity_chase,ebay_chase,set_popularity_chase
0,151,0,Double Rare,0.010417,Alakazam ex,65,True,False,1.08,1.08,0.564376,0.634537,0.811,7.79,1.975880,NaN,NaN,NaN,NaN,NaN,0,0.732368,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.0
1,151,0,Ultra Rare,0.003906,Alakazam ex,188,True,False,7.79,7.79,0.564376,0.661007,0.811,42.88,1.705565,1.975880,NaN,NaN,NaN,NaN,0,2.173615,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,0.0,0.0
2,151,0,Special Illustration Rare,0.004464,Alakazam ex,201,True,False,42.88,42.88,0.564376,0.670165,0.811,1.14,-3.627377,1.705565,3.681444,NaN,NaN,NaN,0,3.781459,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0.0,0.0,0.0
3,151,1,Double Rare,0.010417,Alakazam ex,65,True,False,1.08,1.14,0.564376,0.634537,0.811,8.38,1.994820,-3.627377,-1.921813,0.054067,0.055556,3.159901,1,0.760806,0.054067,NaN,NaN,17.25,NaN,0.066087,NaN,NaN,3,0.0,0.0,0.0
4,151,1,Ultra Rare,0.003906,Alakazam ex,188,True,False,7.79,8.38,0.564376,0.661007,0.811,58.48,1.942837,1.994820,-1.632558,0.073007,0.075738,3.165782,1,2.238580,0.073007,NaN,NaN,17.27,NaN,0.485235,NaN,NaN,4,0.0,0.0,0.0


In [5]:
FEATURES = [
    "rarity",
    "is_pokemon",
    "is_chase",
    "pull_rate",

    "popularity_index",
    "ebay_index",
    "set_popularity_index",

    "log_current_price",
    "return_since_release",
    "card_age",
    "history_length",

    "log_return_1m",
    "log_return_3m",
    "log_return_6m",
    "momentum_3m",
    "momentum_6m",

    "price_vs_mean_3m",
    "price_vs_mean_6m",

    "volatility_3m",
    "volatility_6m",

    "popularity_chase",
    "ebay_chase",
    "set_popularity_chase",
]

In [6]:
for col in FEATURES:
    df[col] = df[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

In [7]:
def time_split(df, test_fraction=0.2):

    times = sorted(df["t"].unique())

    split = int(
        len(times) * (1 - test_fraction)
    )

    train_times = times[:split]
    test_times = times[split:]

    train = df[
        df["t"].isin(train_times)
    ].copy()

    test = df[
        df["t"].isin(test_times)
    ].copy()

    train = train[
        np.isfinite(train["next_log_return"])
    ]

    test = test[
        np.isfinite(test["next_log_return"])
    ]

    return train, test

In [8]:
train, test = time_split(df)

print(
    "Train:",
    train["t"].min(),
    "→",
    train["t"].max()
)

print(
    "Test:",
    test["t"].min(),
    "→",
    test["t"].max()
)

print("Train rows:", len(train))
print("Test rows :", len(test))

Train: 0 → 9

Test: 10 → 12

Train rows: 20163

Test rows : 4292

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


categorical_features = [
    "rarity"
]

numeric_features = [
    "is_pokemon",
    "is_chase",
    "pull_rate",

    "popularity_index",
    "ebay_index",
    "set_popularity_index",

    "log_current_price",
    "return_since_release",
    "card_age",
    "history_length",

    "log_return_1m",
    "log_return_3m",
    "log_return_6m",
    "momentum_3m",
    "momentum_6m",

    "price_vs_mean_3m",
    "price_vs_mean_6m",

    "volatility_3m",
    "volatility_6m",

    "popularity_chase",
    "ebay_chase",
    "set_popularity_chase"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [10]:
X_train = train[FEATURES]
y_train = train["next_log_return"]

X_test = test[FEATURES]
y_test = test["next_log_return"]

In [11]:
X_train_encoded = preprocessor.fit_transform(
    X_train
)

X_test_encoded = preprocessor.transform(
    X_test
)

In [12]:
def train_catboost_quantile(
    X_train,
    y_train,
    alpha
):
    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={alpha}",
        iterations=1000,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train
    )

    return model

In [13]:
cat_models = {}

for alpha in [0.10, 0.25, 0.50, 0.75, 0.90]:

    print(
        f"Training CatBoost q={alpha}"
    )

    cat_models[alpha] = train_catboost_quantile(
        X_train_encoded,
        y_train,
        alpha
    )

Training CatBoost q=0.1

Training CatBoost q=0.25

Training CatBoost q=0.5

Training CatBoost q=0.75

Training CatBoost q=0.9

In [14]:
def evaluate_quantile_models(
    models,
    X_test,
    test
):

    current_price = (
        test["current_price"]
        .to_numpy()
    )

    actual_price = (
        test["next_price"]
        .to_numpy()
    )

    actual_return = (
        test["next_log_return"]
        .to_numpy()
    )

    predictions = {}

    for alpha, model in models.items():

        predictions[alpha] = model.predict(
            X_test
        )

    p10_return = predictions[0.10]
    p25_return = predictions[0.25]
    p50_return = predictions[0.50]
    p75_return = predictions[0.75]
    p90_return = predictions[0.90]

    p10_price = (
        current_price *
        np.exp(p10_return)
    )

    p25_price = (
        current_price *
        np.exp(p25_return)
    )

    p50_price = (
        current_price *
        np.exp(p50_return)
    )

    p75_price = (
        current_price *
        np.exp(p75_return)
    )

    p90_price = (
        current_price *
        np.exp(p90_return)
    )

    mae = mean_absolute_error(
        actual_price,
        p50_price
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_price,
            p50_price
        )
    )

    log_mae = np.mean(
        np.abs(
            actual_return -
            p50_return
        )
    )

    log_rmse = np.sqrt(
        np.mean(
            (
                actual_return -
                p50_return
            ) ** 2
        )
    )

    directional_accuracy = np.mean(
        np.sign(actual_return) ==
        np.sign(p50_return)
    )

    coverage_50 = np.mean(
        (actual_price >= p25_price) &
        (actual_price <= p75_price)
    )

    coverage_90 = np.mean(
        (actual_price >= p10_price) &
        (actual_price <= p90_price)
    )

    def pinball_loss(y, pred, alpha):

        error = y - pred

        return np.mean(
            np.maximum(
                alpha * error,
                (alpha - 1) * error
            )
        )

    pinball = {}

    for alpha in predictions:

        pinball[alpha] = pinball_loss(
            actual_return,
            predictions[alpha],
            alpha
        )

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "Log MAE": log_mae,
        "Log RMSE": log_rmse,
        "Directional accuracy": directional_accuracy,
        "50% coverage": coverage_50,
        "90% coverage": coverage_90,
        "Pinball losses": pinball
    }

    predictions_df = test[
        [
            "set",
            "card",
            "t",
            "current_price",
            "next_price"
        ]
    ].copy()

    predictions_df["p10"] = p10_price
    predictions_df["p25"] = p25_price
    predictions_df["p50"] = p50_price
    predictions_df["p75"] = p75_price
    predictions_df["p90"] = p90_price

    return metrics, predictions_df

In [15]:
cat_metrics, cat_predictions = evaluate_quantile_models(
    cat_models,
    X_test_encoded,
    test
)

print(cat_metrics)

{
    'MAE': 4.263574766099056,
    'RMSE': np.float64(21.43169322359655),
    'Log MAE': np.float64(0.1449846080311366),
    'Log RMSE': np.float64(0.2186069634856431),
    'Directional accuracy': np.float64(0.7760950605778192),
    '50% coverage': np.float64(0.3417986952469711),
    '90% coverage': np.float64(0.619990680335508),
    'Pinball losses': {
        0.1: np.float64(0.044467386691646356),
        0.25: np.float64(0.06294618279192589),
        0.5: np.float64(0.0724923040155683),
        0.75: np.float64(0.05738615294503109),
        0.9: np.float64(0.048185413215614925)
    }
}

In [16]:
from lightgbm import LGBMRegressor

def train_lgbm_quantile(
    X_train,
    y_train,
    alpha
):

    model = LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=5,
        min_child_samples=30,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train
    )

    return model

In [17]:
lgbm_models = {}

for alpha in [0.10, 0.25, 0.50, 0.75, 0.90]:

    print(
        f"Training LightGBM q={alpha}"
    )

    lgbm_models[alpha] = train_lgbm_quantile(
        X_train_encoded,
        y_train,
        alpha
    )

Training LightGBM q=0.1

Training LightGBM q=0.25

Training LightGBM q=0.5

Training LightGBM q=0.75

Training LightGBM q=0.9

In [18]:
lgbm_metrics, lgbm_predictions = (
    evaluate_quantile_models(
        lgbm_models,
        X_test_encoded,
        test
    )
)

print(lgbm_metrics)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/skle

{
    'MAE': 4.133201451716531,
    'RMSE': np.float64(32.656670085594065),
    'Log MAE': np.float64(0.1351774862755373),
    'Log RMSE': np.float64(0.20484521319830293),
    'Directional accuracy': np.float64(0.7739981360671015),
    '50% coverage': np.float64(0.3688257222739981),
    '90% coverage': np.float64(0.6537744641192917),
    'Pinball losses': {
        0.1: np.float64(0.038491196125234155),
        0.25: np.float64(0.05715741449411951),
        0.5: np.float64(0.06758874313776865),
        0.75: np.float64(0.05765788492583609),
        0.9: np.float64(0.03567314182226135)
    }
}

In [ ]:
test_results = lgbm_predictions.copy()

test_results["age"] = test["card_age"].values

test_results["covered_90"] = (
    (test_results["next_price"] >= test_results["p10"]) &
    (test_results["next_price"] <= test_results["p90"])
)

test_results["age_group"] = pd.cut(
    test_results["age"],
    bins=[0, 3, 6, 12, 24, np.inf],
    labels=[
        "0-3m",
        "3-6m",
        "6-12m",
        "12-24m",
        "24m+"
    ]
)

print(
    test_results
    .groupby("age_group", observed=True)["covered_90"]
    .mean()
)

age_group
6-12m    0.653774
Name: covered_90, dtype: float64

In [23]:
print("Age distribution:")
print(test["card_age"].describe())

print("\nAge values:")
print(test["card_age"].value_counts().sort_index())

Age distribution:

count    4292.000000
mean       10.690820
std         0.685557
min        10.000000
25%        10.000000
50%        11.000000
75%        11.000000
max        12.000000
Name: card_age, dtype: float64

Age values:

card_age
10    1877
11    1865
12     550
Name: count, dtype: int64